In [36]:

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
from langchain.tools import tool
from langchain.agents import initialize_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


In [2]:
load_dotenv(override=True)

True

In [4]:
loader = PyPDFLoader("cv.pdf")

In [8]:
tokennizer = tiktoken.encoding_for_model('gpt-4o-mini')

In [9]:
print(tokennizer.name)

o200k_base


In [12]:
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name= tokennizer.name,
    chunk_size=300,
    chunk_overlap=20
)

In [14]:
chunks = loader.load_and_split(splitter)

In [15]:
print(len(chunks))

3


In [16]:
print(chunks[0])

page_content='Najoua Mouaddab\nÉlève-Ingénieur d’État en Informatique et Intelligence Artificielle\n/envel⌢pen.mouaddab@hestim.ma |♂¶obile+212 60 586 9073 |♂¶ap-¶arkerCasablanca, Maroc |/linkedinin/najoua-mouaddab\nProfil\nÉlève-ingénieur d’État en Informatique et IA, passionnée par la Data Science et le développement Full Stack. Curieuse et\norientée résultats, je conçois des solutions performantes et je recherche un stage pour des projets innovants à fort impact.\nForma tion\nHESTIM, Casablanca 2024 – Présent\n2ème année Cycle d’Ingénierie d’État Informatique et IA\nISTA NTIC 2 2022 – 2024\nTechnicien Spécialisé en Développement Digital (Full Stack)\nUniversité Hassan II des Sciences, Casablanca2020 – 2022\nSciences Physiques\nLycée Othman Ben Affan, Casablanca2019 – 2020\nBaccalauréat Sciences Physiques et Chimiques – Option Français\nExpériences Professionnelles\nStagiaire Développement – Pro Skills ConsultingAvril – Juillet 2025\n•Développement d’un portail de gestion de projets.\

In [18]:
embedding_model=OpenAIEmbeddings()

In [22]:
vector_stor = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="cv_data_collection"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [23]:
retriever = vector_stor.as_retriever(kwargs={"fetch_k": 10})

In [ ]:
@tool
def retriever_tool(query : str) ->str:
    """
    permet de chercher des info sur des candidat:
    nom,prenom ,diplomeexperience
    competence
    """
    relevent_chunks = retriever.invoke(query)
    context_list = [d.page_content for d in relevent_chunks]
    context = ". ".join(context_list)
    return context

In [47]:
loader = PyPDFLoader("cv.pdf")
docs = loader.load()

tokenizer = tiktoken.get_encoding("cl100k_base")

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name=tokenizer.name,
    chunk_size=300,
    chunk_overlap=20
)

chunks = splitter.split_documents(docs)

embedding_model = OpenAIEmbeddings()

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="cv_data_collection"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [56]:
@tool
def get_company_infos(company_name :str):
    """consulter des info sur entreprise donne
    """
    return {
        "company_name": company_name,
        "location": "Paris, France",
        "industry": "Technology",
        "size": "500-1000 employees",
        "founded": "2010",
        
    }

In [48]:
from langchain.agents import initialize_agent, AgentType

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 4
agent = initialize_agent(
    tools=[retriever_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [57]:
resp = agent.run("nom, prenom, diplomes, les info sur entreprise")





> Entering new AgentExecutor chain...
I need to gather information about the person's name, surname, degrees, and details about the company. I will use the retriever tool to find this information in the CV. 

Action: retriever_tool  
Action Input: "nom, prenom, diplomes, les info sur entreprise"  
Observation: Expériences Professionnelles
Stagiaire Développement – Pro Skills ConsultingAvril – Juillet 2025
•Développement d’un portail de gestion de projets.
•Modélisation de la base de données relationnelle.
•Gestion des projets, tâches et utilisateurs.
•Tableau de bord interactif et reporting PDF.
•Tests et amélioration continue.
•Travail en équipe.
Programme certifié – COIP L’Heure Joyeuse2019 – 2020
•Renforcement des capacités et projet citoyen.
•Modules : Life Skills, entrepreneuriat, engagement civique, communication, informatique.
Projets Académiques
•Gestion Clubs Sportifs (MERN) : suivi performances et adhérents.
•E-Learning (Angular/Spring Boot) : recommandation de cours.
•Agen

In [58]:
print(resp)

- **Nom**: Mouaddab
- **Prénom**: Najoua
- **Diplômes**:
  - Élève-Ingénieur d’État en Informatique et Intelligence Artificielle (HESTIM, Casablanca, 2024 – Présent)
  - Technicien Spécialisé en Développement Digital (Full Stack) (ISTA NTIC 2, 2022 – 2024)
  - Sciences Physiques (Université Hassan II des Sciences, 2020 – 2022)
  - Baccalauréat Sciences Physiques et Chimiques – Option Français (Lycée Othman Ben Affan, 2019 – 2020)
- **Informations sur l'entreprise**: Pro Skills Consulting (Stagiaire Développement, Avril – Juillet 2025)
  - Développement d’un portail de gestion de projets
  - Modélisation de la base de données relationnelle
  - Gestion des projets, tâches et utilisateurs
  - Tableau de bord interactif et reporting PDF
  - Tests et amélioration continue
  - Travail en équipe


In [53]:
from IPython.display import Markdown

In [55]:
from IPython.display import display, Markdown

display(Markdown(resp))

Nom: Najoua, Prénom: Mouaddab, Diplômes: Élève-Ingénieur d’État en Informatique et Intelligence Artificielle, Technicien Spécialisé en Développement Digital (Full Stack), Baccalauréat Sciences Physiques et Chimiques – Option Français.